# peekabook-crs-test5 결과 분석

wandb 프로젝트 `peekabook-crs-test5`에서 실험 결과를 불러와
NDCG@3,5,10 / Hit-rate@3,5,10 / 평균 점수 / 평균 비용을 계산

## test4 대비 변경 사항

- **장르 필터 전략 변경**: `extract_genre_node` → `extract_genre_node_v2`
  - 기존: 대분류 *최대* 2개, 중분류 *최대* 3개 (LLM이 더 적게 선택할 수 있음)
  - 변경: 대분류 **반드시 3개**, 중분류 **반드시 5개** 선택 강제
  - 목적: 필터가 너무 좁게 잡혀 관련 도서가 검색 대상에서 제외되는 문제 완화
- wandb tag: `mandatory_genre_filter`

**전제**: 각 런에 `book_1_score` ~ `book_10_score` (0 또는 1) 가 기록되어 있어야 함

In [4]:
import math
import pandas as pd
import wandb

# ── 설정 ─────────────────────────────────────────────────────────────────────
WANDB_PROJECT  = "jjeong3150-aiffel/peekabook-crs-test7-2"
N_BOOKS        = 10

# 비용 계산용 단가 (USD per 1M tokens) — 사용 모델에 맞게 수정
MODEL_COSTS = {
    "gpt-4o-mini":       (0.15,  0.60),
    "gpt-4o":            (2.50, 10.00)
}
ACTIVE_MODEL = "gpt-4o-mini"

# ── Sweep 선택 ────────────────────────────────────────────────────────────────
# None 이면 프로젝트 전체 런 로드
# sweep ID를 지정하면 해당 sweep 런만 로드 (아래 셀에서 목록 확인 가능)
# SWEEP_ID = None
SWEEP_ID = "23inzx6f"

In [5]:
# ── 프로젝트 내 Sweep 목록 조회 ──────────────────────────────────────────────
# SWEEP_ID를 고르기 전에 이 셀을 먼저 실행해서 ID를 확인하세요

api = wandb.Api()
sweeps = api.project(WANDB_PROJECT.split("/")[1], entity=WANDB_PROJECT.split("/")[0]).sweeps()

sweep_info = pd.DataFrame([{
    "sweep_id":           s.id,
    "name":               s.name,
    "state":              s.state,
    "expected_run_count": s.expected_run_count,
    "created_at":         s.created_at,
} for s in sweeps]).sort_values("created_at", ascending=False)

print(f"총 {len(sweep_info)}개 sweep")
sweep_info

총 4개 sweep


,sweep_id,name,state,expected_run_count,created_at
0,23inzx6f,23inzx6f,FINISHED,90,2026-06-18T16:05:53Z
1,t55wpzq3,t55wpzq3,FINISHED,18,2026-06-18T06:17:25Z
2,6da2ob8m,6da2ob8m,FINISHED,54,2026-06-18T04:51:39Z
3,jtqq8097,jtqq8097,FINISHED,81,2026-06-18T03:03:32Z


In [6]:
# ── 지표 계산 함수 ─────────────────────────────────────────────────────────

def dcg_at_k(scores: list, k: int) -> float:
    return sum(r / math.log2(i + 2) for i, r in enumerate(scores[:k]))

def ndcg_at_k(scores: list, k: int) -> float:
    dcg  = dcg_at_k(scores, k)
    idcg = dcg_at_k(sorted(scores, reverse=True), k)
    return round(dcg / idcg, 4) if idcg > 0 else 0.0

def hit_rate_at_k(scores: list, k: int) -> int:
    return 1 if any(s > 0 for s in scores[:k]) else 0

def compute_metrics(ranked_scores: list) -> dict:
    n = len(ranked_scores)
    return {
        "ndcg@3":        ndcg_at_k(ranked_scores, 3),
        "ndcg@5":        ndcg_at_k(ranked_scores, 5),
        "ndcg@10":       ndcg_at_k(ranked_scores, 10),
        "hr@3":          hit_rate_at_k(ranked_scores, 3),
        "hr@5":          hit_rate_at_k(ranked_scores, 5),
        "hr@10":         hit_rate_at_k(ranked_scores, 10),
        "mean_score@3":  round(sum(ranked_scores[:3])  / 3,  4) if n >= 3  else 0.0,
        "mean_score@5":  round(sum(ranked_scores[:5])  / 5,  4) if n >= 5  else 0.0,
        "mean_score@10": round(sum(ranked_scores[:10]) / 10, 4) if n >= 10 else 0.0,
    }

def tokens_to_cost(input_tokens: int, output_tokens: int, model: str = ACTIVE_MODEL) -> float:
    in_rate, out_rate = MODEL_COSTS.get(model, MODEL_COSTS["gpt-4o-mini"])
    return (input_tokens * in_rate + output_tokens * out_rate) / 1_000_000

In [7]:
# ── wandb 런 로드 ──────────────────────────────────────────────────────────
# SWEEP_ID가 지정되면 해당 sweep 런만, None이면 프로젝트 전체 런 로드

if SWEEP_ID is not None:
    runs = api.sweep(f"{WANDB_PROJECT}/{SWEEP_ID}").runs
    print(f"Sweep {SWEEP_ID} 런 로드")
else:
    runs = api.runs(WANDB_PROJECT)
    print("프로젝트 전체 런 로드")

records = []
for run in runs:
    s   = run.summary
    cfg = run.config

    scores = []
    for i in range(1, N_BOOKS + 1):
        v = s.get(f"book_{i}_score")
        if v is None:
            break
        scores.append(int(v))

    if not scores:
        continue

    cost = tokens_to_cost(
        int(s.get("total_input_tokens",  0)),
        int(s.get("total_output_tokens", 0)),
    )

    records.append({
        "run_id":            run.id,
        "run_name":          run.name,
        "sweep_id":          run.sweep.id if run.sweep else None,
        "persona_name":      cfg.get("persona_name",     s.get("persona_name", "")),
        "query_transform":   cfg.get("query_transform",  ""),
        "use_genre_filter":  cfg.get("use_genre_filter", ""),
        "collection_name":   cfg.get("collection_name",  ""),
        "session_id":        s.get("session_id", ""),
        "n_books_evaluated": len(scores),
        **compute_metrics(scores),
        "total_input_tokens":  int(s.get("total_input_tokens",  0)),
        "total_output_tokens": int(s.get("total_output_tokens", 0)),
        "estimated_cost_usd":  round(cost, 6),
    })

df = pd.DataFrame(records)
print(f"로드된 런: {len(df)}개")
df.head()

Sweep 23inzx6f 런 로드
로드된 런: 54개


,run_id,run_name,sweep_id,persona_name,query_transform,use_genre_filter,collection_name,session_id,n_books_evaluated,ndcg@3,...,ndcg@10,hr@3,hr@5,hr@10,mean_score@3,mean_score@5,mean_score@10,total_input_tokens,total_output_tokens,estimated_cost_usd
0,0mrhaf0y,sage-sweep-1,23inzx6f,A_최재원,none,False,books_intro_48k,a88cd5b2,10,0.2346,...,0.6340,1,1,1,0.3333,0.2,0.6,10705,1875,0.002731
1,xaqs2krb,cerulean-sweep-2,23inzx6f,A_최재원,none,False,books_intro_48k,3452be16,10,0.5307,...,0.7525,1,1,1,0.6667,0.6,0.5,12458,1975,0.003054
2,r64bqnk7,lively-sweep-3,23inzx6f,A_최재원,none,False,books_intro_48k,ddf81690,10,1.0000,...,0.9783,1,1,1,1.0000,0.8,0.8,10561,1741,0.002629
3,li1hee47,soft-sweep-4,23inzx6f,A_최재원,none,False,books_intro_48k,6265b5f3,10,0.4693,...,0.8281,1,1,1,0.3333,0.4,0.6,13091,2075,0.003209
4,v3hciye1,brisk-sweep-5,23inzx6f,A_최재원,none,False,books_intro_48k,5ca05877,10,0.4693,...,0.7946,1,1,1,0.3333,0.4,0.6,12215,1949,0.003002


In [8]:
# ── 조건별 집계 ────────────────────────────────────────────────────────────

GROUP_COLS = ["collection_name", "use_genre_filter"]

summary = (
    df.groupby(GROUP_COLS)
    .agg(
        runs              = ("run_id",              "count"),
        ndcg_3            = ("ndcg@3",              "mean"),
        ndcg_5            = ("ndcg@5",              "mean"),
        ndcg_10           = ("ndcg@10",             "mean"),
        hr_3              = ("hr@3",                "mean"),
        hr_5              = ("hr@5",                "mean"),
        hr_10             = ("hr@10",               "mean"),
        mean_score_top3   = ("mean_score@3",        "mean"),
        mean_score_top5   = ("mean_score@5",        "mean"),
        mean_score_top10  = ("mean_score@10",       "mean"),
        avg_cost_usd      = ("estimated_cost_usd",  "mean"),
    )
    .round(4)
    .reset_index()
)

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", "{:.4f}".format)
summary

,collection_name,use_genre_filter,runs,ndcg_3,ndcg_5,ndcg_10,hr_3,hr_5,hr_10,mean_score_top3,mean_score_top5,mean_score_top10,avg_cost_usd
0,books_intro_48k,False,27,0.5571,0.6060,0.7989,0.9259,1.0000,1.0000,0.5185,0.5037,0.4481,0.0028
1,books_pub_review_46k,False,27,0.5578,0.5822,0.7829,0.9630,1.0000,1.0000,0.5309,0.5037,0.4481,0.0028


In [9]:
from scipy import stats

METRIC = "ndcg@3"  # 비교할 지표

# ── 비교 그룹 분리 ─────────────────────────────────────────────────────────

# 1) collection 비교 (intro vs merged), genre_filter 조건 고정
for genre_filter in [True, False]:
    subset = df[df["use_genre_filter"] == genre_filter]
    a = subset[subset["collection_name"] == "books_intro_48k"][METRIC]
    b = subset[subset["collection_name"] == "books_merged_48k"][METRIC]

    t, p = stats.ttest_ind(a, b)
    print(f"[collection 비교 | genre_filter={genre_filter}]")
    print(f"  intro: mean={a.mean():.4f}, n={len(a)}")
    print(f"  merged: mean={b.mean():.4f}, n={len(b)}")
    print(f"  t={t:.4f}, p={p:.4f} {'✅ 유의' if p < 0.05 else '❌ 비유의'}\n")

# 2) genre_filter 비교 (True vs False), collection 조건 고정
for collection in ["books_intro_48k", "books_merged_48k"]:
    subset = df[df["collection_name"] == collection]
    a = subset[subset["use_genre_filter"] == False][METRIC]
    b = subset[subset["use_genre_filter"] == True][METRIC]

    t, p = stats.ttest_ind(a, b)
    print(f"[genre_filter 비교 | collection={collection}]")
    print(f"  filter=False: mean={a.mean():.4f}, n={len(a)}")
    print(f"  filter=True:  mean={b.mean():.4f}, n={len(b)}")
    print(f"  t={t:.4f}, p={p:.4f} {'✅ 유의' if p < 0.05 else '❌ 비유의'}\n")

[collection 비교 | genre_filter=True]
  intro: mean=nan, n=0
  merged: mean=nan, n=0
  t=nan, p=nan ❌ 비유의

[collection 비교 | genre_filter=False]
  intro: mean=0.5571, n=27
  merged: mean=nan, n=0
  t=nan, p=nan ❌ 비유의

[genre_filter 비교 | collection=books_intro_48k]
  filter=False: mean=0.5571, n=27
  filter=True:  mean=nan, n=0
  t=nan, p=nan ❌ 비유의

[genre_filter 비교 | collection=books_merged_48k]
  filter=False: mean=nan, n=0
  filter=True:  mean=nan, n=0
  t=nan, p=nan ❌ 비유의



/tmp/ipykernel_1241461/2890557160.py:13: SmallSampleWarning: One or more sample arguments is too small; all returned values will be NaN. See documentation for sample size requirements.
  t, p = stats.ttest_ind(a, b)
/tmp/ipykernel_1241461/2890557160.py:13: SmallSampleWarning: One or more sample arguments is too small; all returned values will be NaN. See documentation for sample size requirements.
  t, p = stats.ttest_ind(a, b)
/tmp/ipykernel_1241461/2890557160.py:25: SmallSampleWarning: One or more sample arguments is too small; all returned values will be NaN. See documentation for sample size requirements.
  t, p = stats.ttest_ind(a, b)
/tmp/ipykernel_1241461/2890557160.py:25: SmallSampleWarning: One or more sample arguments is too small; all returned values will be NaN. See documentation for sample size requirements.
  t, p = stats.ttest_ind(a, b)


In [6]:
# # ── 페르소나별 평균 ────────────────────────────────────────────────────────

# (
#     df.groupby("persona_name")[metric_cols]
#     .mean()
#     .round(4)
# )

In [7]:
# # ── 런별 상세 (정렬: ndcg@10 내림차순) ────────────────────────────────────

# detail_cols = ["persona_name", "query_transform", "use_genre_filter",
#                "ndcg@3", "ndcg@5", "ndcg@10", "hr@3", "hr@5", "hr@10",
#                "mean_score", "estimated_cost_usd", "session_id"]
# df[detail_cols].sort_values("ndcg@10", ascending=False).reset_index(drop=True)